**TE Differential Expression for Individual-Gene Cohorts**
This notebook runs the same limma framework as group-level analyses, but loops over each individual gene/cohort.

For each cohort it writes:
- <gene_set_label>_TE_subfamily_matched_limma_results.csv
- <gene_set_label>_TE_family_matched_limma_results.csv
- <gene_set_label>_TE_subfamily_covariate_limma_results.csv
- <gene_set_label>_TE_family_covariate_limma_results.csv
- <gene_set_label>_TE_ERV_LTR_matched_limma_results.csv
- <gene_set_label>_TE_DE_significance_summary.csv
- <gene_set_label>_candidate_direction_by_tumour_type.csv
- <gene_set_label>_TE_subfamily_volcano.png

The primary model is still: ~ matched_set + alteration_status
- This compares altered cases against their matched WT controls while accounting for the case-control matched set.

In [ ]:
# Load libraries and set paths

library(dplyr)
library(readr)
library(tibble)
library(Biobase)
library(limma)
library(stringr)
library(purrr)
library(ggplot2)

project_root <- "/home/kennes38/ResearchProject"
notebook_dir <- file.path(project_root, "04_individualgenes_TE_differential_expression")
dir.create(notebook_dir, showWarnings = FALSE, recursive = TRUE)
setwd(notebook_dir)

match_dir <- "../04_individualgenes_WTmatching_tumouronly"

# Extracted REdiscoverTE data folder
tedata_root <- "../TEdata_resources"

te_de_output_dir <- "."

dir.exists(match_dir)
dir.exists(tedata_root)
dir.exists(te_de_output_dir)

In [ ]:
# Define cohorts to process

analysis_definitions <- tibble::tribble(
  ~gene_set_label, ~gene_set_group, ~display_name, ~genes,
  "kdm6a_loss",   "histone_modifier", "KDM6A loss",   list(c("KDM6A")),
  "kdm6b_loss",   "histone_modifier", "KDM6B loss",   list(c("KDM6B")),
  "nsd1_loss",    "histone_modifier", "NSD1 loss",    list(c("NSD1")),
  "nsd2_loss",    "histone_modifier", "NSD2 loss",    list(c("NSD2")),
  "setd2_loss",   "histone_modifier", "SETD2 loss",   list(c("SETD2")),
  "kdm6a_b_loss", "histone_modifier", "KDM6A/B loss", list(c("KDM6A", "KDM6B")),
  "nsd1_2_loss",  "histone_modifier", "NSD1/2 loss",  list(c("NSD1", "NSD2")),
  "arid1a_loss",   "swi_snf", "ARID1A loss",   list(c("ARID1A")),
  "arid1b_loss",   "swi_snf", "ARID1B loss",   list(c("ARID1B")),
  "arid1a_b_loss", "swi_snf", "ARID1A/B loss", list(c("ARID1A", "ARID1B")),
  "smarca4_loss",  "swi_snf", "SMARCA4 loss",  list(c("SMARCA4")),
  "smarcb1_loss", "swi_snf", "SMARCB1 loss", list(c("SMARCB1")),
  "pbrm1_loss",   "swi_snf", "PBRM1 loss",   list(c("PBRM1"))
)

In [ ]:
# Load REdiscoverTE ExpressionSet and annotation

rds_hits <- list.files(
  tedata_root,
  pattern = "eset_TCGA_TE_intergenic_logCPM\\.RDS$",
  recursive = TRUE,
  full.names = TRUE
)

stopifnot(length(rds_hits) == 1)
dat <- readRDS(rds_hits[1])

# Full REdiscoverTE log2CPM matrix, rows - TE features, columns - samples
expr_full <- exprs(dat)

annotation_hits <- list.files(
  tedata_root,
  pattern = "repName_repClass_repFam_TE\\.RDS$",
  recursive = TRUE,
  full.names = TRUE
)

stopifnot(length(annotation_hits) == 1)
annotation_raw <- readRDS(annotation_hits[1])

annotation <- annotation_raw %>%
  as.data.frame() %>%
  tibble::rownames_to_column("feature_id_from_rownames") %>%
  as_tibble()

if (!"feature_id" %in% colnames(annotation)) {
  if (any(annotation$feature_id_from_rownames %in% rownames(expr_full))) {
    annotation <- annotation %>% mutate(feature_id = feature_id_from_rownames)
  } else if ("repName" %in% colnames(annotation) && any(annotation$repName %in% rownames(expr_full))) {
    annotation <- annotation %>% mutate(feature_id = repName)
  } else {
    stop("Could not identify feature_id in the TE annotation.")
  }
}

annotation <- annotation %>%
  rename(
    TE_subfamily = repName,
    TE_class = repClass,
    TE_family = repFam
  ) %>%
  select(feature_id, TE_subfamily, TE_class, TE_family, everything())

cat("Expression matrix dimensions:", dim(expr_full), "\n")
cat("Annotated features overlapping expression matrix:", sum(annotation$feature_id %in% rownames(expr_full)), "\n")

In [ ]:
# Set limma and plotting helper functions

run_limma <- function(expression_matrix, metadata, design_formula, coef_name, feature_level) {
  # model.matrix converts metadata columns into covariates for limma
  design <- model.matrix(design_formula, data = metadata)

  # lmFit fits one linear model per TE feature/family
  fit <- lmFit(expression_matrix, design)

  # eBayes borrows information across all rows to stabilise variance estimates
  fit <- eBayes(fit, trend = TRUE)

  topTable(
    fit,
    coef = coef_name,
    number = Inf,
    adjust.method = "BH",
    sort.by = "P"
  ) %>%
    rownames_to_column("feature_id") %>%
    as_tibble() %>%
    mutate(feature_level = feature_level)
}

make_family_expression <- function(expr, annotation, sample_ids) {
  expr_tbl <- as_tibble(expr, rownames = "feature_id", .name_repair = "minimal") %>%
    left_join(annotation, by = "feature_id")

  family_expr_tbl <- expr_tbl %>%
    filter(!is.na(TE_family), TE_family != "") %>%
    select(-feature_id, -TE_subfamily, -TE_class) %>%
    group_by(TE_family) %>%
    summarise(across(where(is.numeric), \(x) mean(x, na.rm = TRUE)), .groups = "drop")

  family_expr <- family_expr_tbl %>%
    column_to_rownames("TE_family") %>%
    as.matrix()

  family_expr[, sample_ids, drop = FALSE]
}

summarise_results <- function(all_results) {
  all_results %>%
    group_by(feature_level) %>%
    summarise(
      n_tested = n(),
      n_fdr_0_05 = sum(adj.P.Val < 0.05, na.rm = TRUE),
      n_fdr_0_10 = sum(adj.P.Val < 0.10, na.rm = TRUE),
      n_nominal_0_05 = sum(P.Value < 0.05, na.rm = TRUE),
      .groups = "drop"
    )
}

direction_by_tumour_type <- function(candidate, analysis_meta, expr) {
  purrr::map_dfr(split(analysis_meta, analysis_meta$tumour_type), function(meta_t) {
    samples <- meta_t$TEdata_full_id
    values <- expr[candidate, samples]

    mean_loss <- mean(values[meta_t$alteration_status == "altered"], na.rm = TRUE)
    mean_wt <- mean(values[meta_t$alteration_status == "WT"], na.rm = TRUE)

    tibble(
      feature_id = candidate,
      tumour_type = as.character(unique(meta_t$tumour_type)),
      n_loss = sum(meta_t$alteration_status == "altered"),
      n_wt = sum(meta_t$alteration_status == "WT"),
      mean_loss = mean_loss,
      mean_wt = mean_wt,
      delta_log2CPM = mean_loss - mean_wt
    )
  })
}

make_volcano <- function(matched_subfamily_results, gene_set_label, display_name) {
  volcano_df <- matched_subfamily_results %>%
    mutate(
      neg_log10_p = -log10(P.Value),
      result = case_when(
        adj.P.Val < 0.05 ~ "FDR < 0.05",
        P.Value < 0.05 ~ "Nominal P < 0.05",
        TRUE ~ "Not significant"
      )
    )

  p <- ggplot(volcano_df, aes(x = logFC, y = neg_log10_p, colour = result)) +
    geom_point(alpha = 0.7, size = 1.8) +
    geom_vline(xintercept = 0, linetype = "dotted", colour = "grey60") +
    geom_hline(yintercept = -log10(0.05), linetype = "dashed", colour = "grey40") +
    scale_colour_manual(
      values = c(
        "FDR < 0.05" = "#D55E00",
        "Nominal P < 0.05" = "#2C6DB2",
        "Not significant" = "grey75"
      )
    ) +
    labs(
      title = "TE subfamily differential expression",
      subtitle = paste0(display_name, " primary tumours vs matched WT primary tumour controls"),
      x = "logFC",
      y = "-log10(P value)",
      colour = "Result"
    ) +
    theme_classic()

  ggsave(
    file.path(te_de_output_dir, paste0(gene_set_label, "_TE_subfamily_volcano.png")),
    p,
    width = 7,
    height = 6,
    dpi = 300
  )

  p
}

In [ ]:
# Set helper function to analyse one cohort

analyse_one_cohort <- function(def_row, min_cases = 3) {
  gene_set_label <- def_row$gene_set_label
  display_name <- def_row$display_name
  message("===== ", gene_set_label, " =====")

  case_control_file <- file.path(
    match_dir,
    paste0(gene_set_label, "_case_control_matched_table_cancer_sex_primary_tumour_only.csv")
  )

  if (!file.exists(case_control_file)) {
    warning("Missing matched case-control file for ", gene_set_label)
    return(tibble(gene_set = gene_set_label, status = "missing_case_control_file"))
  }

  case_control_tbl <- read_csv(case_control_file, show_col_types = FALSE)

  if (nrow(case_control_tbl) == 0 || n_distinct(case_control_tbl$case_sample_id_16) < min_cases) {
    warning("Skipping ", gene_set_label, ": fewer than ", min_cases, " matched cases")
    return(tibble(gene_set = gene_set_label, status = "too_few_cases"))
  }

  case_meta <- case_control_tbl %>%
    transmute(
      TEdata_full_id = case_TEdata_full_id,
      sample_id_16 = case_sample_id_16,
      patient_id = case_patient_id,
      sample_type_code = case_sample_type_code,
      matched_set = case_sample_id_16,
      alteration_status = "altered",
      tumour_type = case_project,
      sex = case_sex,
      gene_set,
      gene_set_group,
      gene_set_display,
      genes_tested,
      genes_lost,
      events,
      lof_genes,
      homdel_genes,
      mutation_classes,
      n_genes_lost,
      n_loss_events,
      has_lof,
      has_homdel
    ) %>%
    distinct()

  control_meta <- case_control_tbl %>%
    transmute(
      TEdata_full_id = WT_TEdata_full_id,
      sample_id_16 = WT_sample_id_16,
      patient_id = WT_patient_id,
      sample_type_code = WT_sample_type_code,
      matched_set = case_sample_id_16,
      alteration_status = "WT",
      tumour_type = WT_project,
      sex = WT_sex,
      gene_set = gene_set_label,
      gene_set_group = def_row$gene_set_group,
      gene_set_display = display_name,
      genes_tested = paste(unique(unlist(def_row$genes, use.names = FALSE)), collapse = ";"),
      genes_lost = NA_character_,
      events = NA_character_,
      lof_genes = NA_character_,
      homdel_genes = NA_character_,
      mutation_classes = NA_character_,
      n_genes_lost = 0L,
      n_loss_events = 0L,
      has_lof = FALSE,
      has_homdel = FALSE
    ) %>%
    distinct()

  analysis_meta <- bind_rows(case_meta, control_meta) %>%
    mutate(
      alteration_status = factor(alteration_status, levels = c("WT", "altered")),
      matched_set = factor(matched_set),
      tumour_type = factor(tumour_type),
      sex = factor(sex)
    ) %>%
    arrange(matched_set, alteration_status, TEdata_full_id)

  duplicated_samples <- analysis_meta %>%
    count(TEdata_full_id, sort = TRUE) %>%
    filter(n > 1)

  if (nrow(duplicated_samples) > 0) {
    stop(
      "Duplicate TEdata samples found in ", gene_set_label,
      ". Rerun Notebook 23 after the no-reuse WT matching fix. Example duplicate: ",
      duplicated_samples$TEdata_full_id[1]
    )
  }

  missing_samples <- setdiff(analysis_meta$TEdata_full_id, colnames(expr_full))
  if (length(missing_samples) > 0) {
    stop("Some matched samples are missing from expr_full for ", gene_set_label)
  }

  expr <- expr_full[, analysis_meta$TEdata_full_id, drop = FALSE]
  stopifnot(identical(colnames(expr), analysis_meta$TEdata_full_id))

  family_expr <- make_family_expression(expr, annotation, analysis_meta$TEdata_full_id)
  stopifnot(identical(colnames(family_expr), analysis_meta$TEdata_full_id))

  matched_feature_results <- run_limma(
    expression_matrix = expr,
    metadata = analysis_meta,
    design_formula = ~ matched_set + alteration_status,
    coef_name = "alteration_statusaltered",
    feature_level = "feature"
  )

  matched_subfamily_results <- matched_feature_results %>%
    left_join(annotation, by = "feature_id") %>%
    mutate(feature_level = "subfamily")

  family_results <- run_limma(
    expression_matrix = family_expr,
    metadata = analysis_meta,
    design_formula = ~ matched_set + alteration_status,
    coef_name = "alteration_statusaltered",
    feature_level = "family"
  )

  covariate_meta <- analysis_meta %>%
    filter(!is.na(tumour_type), !is.na(sex), tumour_type != "", sex != "") %>%
    droplevels()

  expr_covariate <- expr[, covariate_meta$TEdata_full_id, drop = FALSE]
  family_expr_covariate <- family_expr[, covariate_meta$TEdata_full_id, drop = FALSE]

  covariate_subfamily_results <- run_limma(
    expression_matrix = expr_covariate,
    metadata = covariate_meta,
    design_formula = ~ alteration_status + tumour_type + sex,
    coef_name = "alteration_statusaltered",
    feature_level = "subfamily_covariate_model"
  ) %>%
    left_join(annotation, by = "feature_id")

  covariate_family_results <- run_limma(
    expression_matrix = family_expr_covariate,
    metadata = covariate_meta,
    design_formula = ~ alteration_status + tumour_type + sex,
    coef_name = "alteration_statusaltered",
    feature_level = "family_covariate_model"
  )

  erv_ltr_features <- annotation %>%
    filter(
      TE_class == "LTR" |
        str_detect(TE_family, regex("ERV|LTR", ignore_case = TRUE)) |
        str_detect(TE_subfamily, regex("ERV|LTR", ignore_case = TRUE))
    ) %>%
    pull(feature_id) %>%
    intersect(rownames(expr))

  erv_ltr_results <- run_limma(
    expression_matrix = expr[erv_ltr_features, , drop = FALSE],
    metadata = analysis_meta,
    design_formula = ~ matched_set + alteration_status,
    coef_name = "alteration_statusaltered",
    feature_level = "ERV_LTR_subfamily"
  ) %>%
    left_join(annotation, by = "feature_id")

  all_results <- bind_rows(
    matched_subfamily_results,
    family_results,
    covariate_subfamily_results,
    covariate_family_results,
    erv_ltr_results
  )

  significance_summary <- summarise_results(all_results) %>%
    mutate(gene_set = gene_set_label, gene_set_display = display_name, .before = 1)

  write_csv(matched_subfamily_results, file.path(te_de_output_dir, paste0(gene_set_label, "_TE_subfamily_matched_limma_results.csv")))
  write_csv(family_results, file.path(te_de_output_dir, paste0(gene_set_label, "_TE_family_matched_limma_results.csv")))
  write_csv(covariate_subfamily_results, file.path(te_de_output_dir, paste0(gene_set_label, "_TE_subfamily_covariate_limma_results.csv")))
  write_csv(covariate_family_results, file.path(te_de_output_dir, paste0(gene_set_label, "_TE_family_covariate_limma_results.csv")))
  write_csv(erv_ltr_results, file.path(te_de_output_dir, paste0(gene_set_label, "_TE_ERV_LTR_matched_limma_results.csv")))
  write_csv(significance_summary, file.path(te_de_output_dir, paste0(gene_set_label, "_TE_DE_significance_summary.csv")))

  candidate_hits <- matched_subfamily_results %>%
    filter(P.Value < 0.05) %>%
    arrange(P.Value) %>%
    slice_head(n = 10)

  make_volcano(matched_subfamily_results, gene_set_label, display_name)

  if (nrow(candidate_hits) > 0) {
    direction_results <- purrr::map_dfr(candidate_hits$feature_id, direction_by_tumour_type, analysis_meta = analysis_meta, expr = expr)
    write_csv(direction_results, file.path(te_de_output_dir, paste0(gene_set_label, "_candidate_direction_by_tumour_type.csv")))
  }

  cat("Matched cases:", n_distinct(case_meta$sample_id_16), "\n")
  cat("WT controls:", n_distinct(control_meta$sample_id_16), "\n")
  print(significance_summary)

  significance_summary %>% mutate(status = "analysed")
}

In [ ]:
# Run differential expression for all cohorts

# min_cases avoids running unstable models for very tiny cohorts
# Can lower this to 2 for exploration, but >=3 is a starting point
min_cases <- 3

definition_rows <- split(analysis_definitions, seq_len(nrow(analysis_definitions)))

all_individual_de_summary <- purrr::map_dfr(
  definition_rows,
  function(def_row) {
    tryCatch(
      analyse_one_cohort(def_row, min_cases = min_cases),
      error = function(e) {
        warning("Differential expression failed for ", def_row$gene_set_label, ": ", conditionMessage(e))
        tibble(
          gene_set = def_row$gene_set_label,
          gene_set_display = def_row$display_name,
          status = "model_failed",
          error_message = conditionMessage(e)
        )
      }
    )
  }
)

write_csv(
  all_individual_de_summary,
  file.path(te_de_output_dir, "individual_gene_TE_DE_combined_significance_summary.csv")
)

all_individual_de_summary %>%
  arrange(gene_set, feature_level)

In [ ]:
# Quick view of the strongest matched subfamily hits

# Combine all matched subfamily result files into one table of top hits
# Useful for quickly seeing whether any individual gene stands out
subfamily_files <- list.files(
  te_de_output_dir,
  pattern = "_TE_subfamily_matched_limma_results\\.csv$",
  full.names = TRUE
)

combined_top_hits <- purrr::map_dfr(subfamily_files, function(f) {
  label <- basename(f) %>% str_replace("_TE_subfamily_matched_limma_results\\.csv$", "")
  read_csv(f, show_col_types = FALSE) %>%
    mutate(gene_set = label, .before = 1)
}) %>%
  arrange(adj.P.Val, P.Value)

write_csv(
  combined_top_hits,
  file.path(te_de_output_dir, "individual_gene_TE_subfamily_matched_limma_results_ALL.csv")
)

combined_top_hits %>%
  select(gene_set, feature_id, TE_class, TE_family, logFC, P.Value, adj.P.Val) %>%
  slice_head(n = 30)